In [ ]:
%pip install lifelines

In [ ]:
# Celda S1 - Carga y armado de la tabla de supervivencia
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

BASE = Path('/Users/ppizam/Claude/Master Thesis')
EV = BASE / 'Desarrollo' / 'Metodologia' / 'Matrix' / 'eventos'
(EV / 'supervivencia').mkdir(exist_ok=True)

VERDE1, VERDE2, GRIS = '#00684A', '#1A5742', '#8A8A8A'

cat = pd.read_csv(EV / 'eventos_atencion_v2_principal_final.csv',
                  keep_default_na=False, na_values=[''])
cx = pd.read_csv(EV / 'eventos_con_precios.csv',
                 keep_default_na=False, na_values=[''])

# unir la censura y variables del catalogo con las covariables de mercado
llave = ['ticker', 'fecha_inicio']
sup = cx.merge(cat[llave + ['censurado', 'z_inicio', 'dias_a_pico', 'menciones_pico']],
               on=llave, how='left', suffixes=('', '_cat'))
sup['evento_observado'] = (~sup['censurado'].astype(bool)).astype(int)  # 1 = muerte observada

# grupos de acoplamiento atencion-precio
def grupo_desfase(d):
    if d >= 2:
        return 'atencion anticipa'      # el precio pica 2+ dias despues de la atencion
    if d <= -2:
        return 'reactivo'               # el precio pico 2+ dias antes
    return 'sincronico'
sup['acoplamiento'] = sup['desfase_precio_dias'].apply(grupo_desfase)

print(f'tabla de supervivencia: {len(sup)} eventos | censurados: {int(sup.censurado.sum())}')
print('\ngrupos de acoplamiento:')
print(sup.acoplamiento.value_counts().to_string())
print('\nduracion por grupo (mediana):')
print(sup.groupby('acoplamiento').duracion_dias.median().to_string())
sup.to_csv(EV / 'supervivencia' / 'tabla_supervivencia.csv', index=False)

In [ ]:
# Celda S2 - Kaplan-Meier global y por grupo de acoplamiento (con prueba log-rank)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

km = KaplanMeierFitter()
km.fit(sup.duracion_dias, event_observed=sup.evento_observado, label='todas las burbujas')
km.plot_survival_function(ax=axes[0], color=VERDE1)
axes[0].set_title('Supervivencia de las burbujas de atención (Kaplan-Meier)',
                  color=VERDE1, fontsize=11, fontweight='bold')
axes[0].set_xlabel('días desde el encendido')
axes[0].set_ylabel('fracción de eventos aún vivos')
axes[0].set_xlim(0, 90)
print(f'mediana de supervivencia global: {km.median_survival_time_:.0f} dias')
print(f'sobreviven a 30 dias: {km.predict(30):.1%} | a 60 dias: {km.predict(60):.1%}')

colores = {'atencion anticipa': VERDE1, 'sincronico': GRIS, 'reactivo': '#B3402A'}
for g, df_g in sup.groupby('acoplamiento'):
    km_g = KaplanMeierFitter()
    km_g.fit(df_g.duracion_dias, event_observed=df_g.evento_observado, label=g)
    km_g.plot_survival_function(ax=axes[1], color=colores[g])
    print(f'{g}: mediana {km_g.median_survival_time_:.0f} dias (n={len(df_g)})')
axes[1].set_title('Por acoplamiento atención-precio', color=VERDE1,
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('días desde el encendido')
axes[1].set_ylabel('')
axes[1].set_xlim(0, 90)

plt.tight_layout()
plt.savefig(EV / 'supervivencia' / 'km_global_y_acoplamiento.png', dpi=150,
            bbox_inches='tight')
plt.show()

# prueba log-rank multivariada: ¿difieren las tres curvas?
lr = multivariate_logrank_test(sup.duracion_dias, sup.acoplamiento, sup.evento_observado)
print(f'\nlog-rank (3 grupos): estadistico {lr.test_statistic:.1f}, p-value {lr.p_value:.2e}')

In [ ]:
# Celda S3 - Log-rank por pares y confirmacion del hallazgo
pares = [('atencion anticipa', 'reactivo'),
         ('sincronico', 'atencion anticipa'),
         ('sincronico', 'reactivo')]
print('log-rank por pares:')
for a, b in pares:
    da = sup[sup.acoplamiento == a]
    db = sup[sup.acoplamiento == b]
    lr = logrank_test(da.duracion_dias, db.duracion_dias,
                      da.evento_observado, db.evento_observado)
    print(f'  {a} vs {b}: estadistico {lr.test_statistic:.1f}, p-value {lr.p_value:.2e}')

# grupo binario para el resto del analisis
sup['desacoplado'] = (sup.acoplamiento != 'sincronico').astype(int)

In [ ]:
# Celda S4 v2 - Cox proporcional: modelo predictivo y descriptivo (trae base_previa_mu)
from lifelines import CoxPHFitter

d = sup.merge(cat[['ticker', 'fecha_inicio', 'base_previa_mu']],
              on=['ticker', 'fecha_inicio'], how='left')

d['log_z_inicio'] = np.log1p(d.z_inicio.clip(lower=0))
d['log_base_previa'] = np.log1p(d.base_previa_mu.clip(lower=0))
d['log_amplitud'] = np.log(d.amplitud.clip(lower=0.1))
d['log_vol_ratio'] = np.log(d.vol_ratio_evento.clip(lower=0.1))
d['anio'] = pd.to_datetime(d.fecha_inicio).dt.year

# Modelo A - PREDICTIVO: solo lo observable el dia del encendido
colsA = ['duracion_dias', 'evento_observado', 'log_z_inicio', 'log_base_previa']
dA = d[colsA].dropna()
coxA = CoxPHFitter()
coxA.fit(dA, duration_col='duracion_dias', event_col='evento_observado')
print('=== MODELO A (predictivo, covariables al encendido) ===')
print(coxA.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']]
      .round(3).to_string())
print(f'concordancia: {coxA.concordance_index_:.3f}')

# Modelo B - DESCRIPTIVO: + caracteristicas realizadas durante el evento
colsB = colsA + ['log_amplitud', 'log_vol_ratio', 'ret_encendido_pico', 'desacoplado']
dB = d[colsB].dropna()
coxB = CoxPHFitter()
coxB.fit(dB, duration_col='duracion_dias', event_col='evento_observado')
print('\n=== MODELO B (descriptivo, + realizadas durante el evento) ===')
print(coxB.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']]
      .round(3).to_string())
print(f'concordancia: {coxB.concordance_index_:.3f}')

coxA.summary.to_csv(EV / 'supervivencia' / 'cox_modelo_A_predictivo.csv')
coxB.summary.to_csv(EV / 'supervivencia' / 'cox_modelo_B_descriptivo.csv')
print('\nresumenes guardados en Matrix/eventos/supervivencia/')

In [ ]:
# Celda S5
from pathlib import Path
MATRIX = Path('/Users/ppizam/Claude/Master Thesis/Desarrollo/Metodologia/Matrix')
exec(open(MATRIX / 'celda_S5.py').read())

In [ ]:
# Celda S6
from pathlib import Path
MATRIX = Path('/Users/ppizam/Claude/Master Thesis/Desarrollo/Metodologia/Matrix')
exec(open(MATRIX / 'celda_S6.py').read())

In [ ]:
# Celda S6b - perfil temporal del HR del desacuerdo
import pandas as pd, numpy as np
from pathlib import Path
SUP = Path('/Users/ppizam/Claude/Master Thesis/Desarrollo/Metodologia/Matrix/eventos/supervivencia')
r = pd.read_csv(SUP / 'cox_ph_interacciones.csv').set_index('covariable').loc['d_duro_lag']
print('HR del desacuerdo por edad de la burbuja:')
for t in [1, 3, 7, 14, 30]:
    print(f'  dia {t:>2}: HR {np.exp(r.coef_principal + r.coef_interaccion * np.log(t)):.2f}')

In [ ]:
exec(open('/Users/ppizam/Claude/Master Thesis/Desarrollo/Metodologia/Matrix/celda_K2.py').read())
exec(open('/Users/ppizam/Claude/Master Thesis/Desarrollo/Metodologia/Matrix/celda_K3.py').read())

In [2]:
# Celda S7 - COX POR POBLACIONES: burbujas nativas vs eventos de earnings
# La capa de identificacion del plan de news (encargo acordado con el director):
# cada encendido del catalogo quedo etiquetado con "hubo anuncio de resultados
# (RDQ de Compustat/WRDS) en +/-2 dias" (eventos_etiqueta_earnings.csv, censal).
# La pregunta que responde esta celda: ¿el hallazgo central (la burbuja muere
# del debate) es propio de las burbujas nativas de la conversacion, o comparte
# mecanica con los eventos guiados por noticia institucional?
# Diseno en tres piezas:
#   (a) descriptivo: Kaplan-Meier y log-rank entre poblaciones;
#   (b) el Cox integrado I2 (especificacion IDENTICA a S5) re-estimado en cada
#       poblacion por separado;
#   (c) el test formal: modelo conjunto I2 + dummy earnings + interacciones
#       D(t-1) x earnings y B(t-1) x earnings (¿difiere el efecto entre
#       poblaciones? la p de la interaccion es el veredicto).
# Los eventos sin etiqueta posible (sin gvkey o sin RDQ; ~20%, ADRs/ETFs/
# extranjeros documentados) se excluyen del contraste y se reportan.
import numpy as np
import pandas as pd
from pathlib import Path
from lifelines import CoxTimeVaryingFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test

BASE = Path('/Users/ppizam/Claude/Master Thesis')
EV = BASE / 'Desarrollo' / 'Metodologia' / 'Matrix' / 'eventos'
SUP = EV / 'supervivencia'

# --- 1. insumos: identicos a S5 + la etiqueta censal -------------------------
vida = pd.read_csv(SUP / 'tabla_startstop_bt.csv', keep_default_na=False, na_values=[''])
cat = pd.read_csv(EV / 'eventos_atencion_v2_principal_final.csv',
                  keep_default_na=False, na_values=[''])
sup = pd.read_csv(SUP / 'tabla_supervivencia.csv', keep_default_na=False, na_values=[''])
sup = sup.merge(cat[['ticker', 'fecha_inicio', 'base_previa_mu']],
                on=['ticker', 'fecha_inicio'], how='left')
sup['evento_id'] = sup.ticker + '_' + sup.fecha_inicio.astype(str)

et = pd.read_csv(EV / 'eventos_etiqueta_earnings.csv', keep_default_na=False, na_values=[''])
et['evento_id'] = et.ticker + '_' + et.fecha_inicio.astype(str)
print('etiqueta censal:', et.etiqueta.value_counts().to_dict())

# derivadas identicas a S5 (misma clip y log para comparabilidad exacta)
sup['log_z_inicio'] = np.log1p(sup.z_inicio.clip(lower=0))
sup['log_base_previa'] = np.log1p(sup.base_previa_mu.clip(lower=0))
sup['log_amplitud'] = np.log(sup.amplitud.clip(lower=0.1))
sup['log_vol_ratio'] = np.log(sup.vol_ratio_evento.clip(lower=0.1))
sup['desacoplado'] = (sup.acoplamiento != 'sincronico').astype(int)

ESTATICAS = ['log_z_inicio', 'log_base_previa', 'log_amplitud', 'log_vol_ratio',
             'ret_encendido_pico', 'desacoplado']
tabla = vida.merge(sup[['evento_id'] + ESTATICAS], on='evento_id', how='inner')
tabla = tabla.dropna(subset=ESTATICAS)
tabla = tabla.merge(et[['evento_id', 'etiqueta']], on='evento_id', how='left')

nucleo = tabla[tabla.etiqueta.isin(['earnings', 'sin_earnings'])].copy()
nucleo['earnings'] = (nucleo.etiqueta == 'earnings').astype(int)
n_ev = nucleo.groupby('etiqueta').evento_id.nunique()
print(f"eventos en el contraste (con mercado completo): "
      f"earnings {n_ev.get('earnings', 0):,} | nativos {n_ev.get('sin_earnings', 0):,} | "
      f"excluidos sin etiqueta: {tabla[~tabla.etiqueta.isin(['earnings','sin_earnings'])].evento_id.nunique():,}")

# --- 2. descriptivo: KM y log-rank -------------------------------------------
dur = nucleo.groupby('evento_id').agg(dur_dias=('stop', 'max'),
                                      muere=('evento_muerte', 'max'),
                                      earnings=('earnings', 'first'))
km = KaplanMeierFitter()
for g, nombre in [(1, 'earnings'), (0, 'nativos')]:
    sub = dur[dur.earnings == g]
    km.fit(sub['dur_dias'], sub['muere'])
    print(f'KM {nombre}: mediana {km.median_survival_time_:.0f} dias | '
          f'sobrevive 30d: {float(km.survival_function_at_times(30).iloc[0]):.1%} (n={len(sub):,})')
lr = logrank_test(dur[dur.earnings == 1]['dur_dias'], dur[dur.earnings == 0]['dur_dias'],
                  dur[dur.earnings == 1]['muere'], dur[dur.earnings == 0]['muere'])
print(f'log-rank earnings vs nativos: p = {lr.p_value:.2e}')

# --- 3. el Cox I2 en cada poblacion ------------------------------------------
DINAMICAS = ['b_duro_lag', 'd_duro_lag', 'sin_direccion_lag', 'log1p_n_lag']
COVS_I2 = DINAMICAS + ['log_z_inicio', 'log_base_previa', 'log_amplitud',
                       'log_vol_ratio', 'ret_encendido_pico', 'desacoplado']
BASE_COLS = ['evento_id', 'start', 'stop', 'evento_muerte']

def ajustar(df, covs, nombre):
    ctv = CoxTimeVaryingFitter(penalizer=0.0)
    ctv.fit(df[BASE_COLS + list(covs)], id_col='evento_id', start_col='start',
            stop_col='stop', event_col='evento_muerte', show_progress=False)
    s = ctv.summary[['coef', 'exp(coef)', 'exp(coef) lower 95%',
                     'exp(coef) upper 95%', 'p']].round(4)
    print(f'\n=== {nombre} ===')
    print(s.to_string())
    return ctv.summary.assign(modelo=nombre)

resumenes = []
resumenes.append(ajustar(nucleo[nucleo.earnings == 0], COVS_I2,
                 'P1 - Cox integrado I2, poblacion NATIVA (sin earnings)'))
resumenes.append(ajustar(nucleo[nucleo.earnings == 1], COVS_I2,
                 'P2 - Cox integrado I2, poblacion EARNINGS'))

# --- 4. test formal: modelo conjunto con interacciones -----------------------
nucleo['d_x_earnings'] = nucleo.d_duro_lag * nucleo.earnings
nucleo['b_x_earnings'] = nucleo.b_duro_lag * nucleo.earnings
resumenes.append(ajustar(nucleo, COVS_I2 + ['earnings', 'd_x_earnings', 'b_x_earnings'],
                 'P3 - conjunto con interacciones (test formal)'))

# --- 5. careo final ----------------------------------------------------------
print('\n=== el desacuerdo D(t-1) por poblacion (referencia I2 global: 1.71) ===')
for r in resumenes[:2]:
    f = r.loc['d_duro_lag']
    print(f"{f['modelo']}: HR {f['exp(coef)']:.3f} "
          f"[{f['exp(coef) lower 95%']:.3f}, {f['exp(coef) upper 95%']:.3f}] p={f['p']:.4f}")
f = resumenes[2].loc['d_x_earnings']
print(f"interaccion D x earnings: coef {f['coef']:.3f} p={f['p']:.4f} "
      f"({'las poblaciones DIFIEREN' if f['p'] < 0.05 else 'sin evidencia de diferencia'})")
f = resumenes[2].loc['b_x_earnings']
print(f"interaccion B x earnings: coef {f['coef']:.3f} p={f['p']:.4f}")
f = resumenes[2].loc['earnings']
print(f"dummy earnings (nivel de riesgo basal): HR {f['exp(coef)']:.3f} p={f['p']:.4f}")

todo = pd.concat(resumenes)
todo.to_csv(SUP / 'cox_por_poblaciones.csv')
print('\nguardado: supervivencia/cox_por_poblaciones.csv')
print('\nlectura: (1) si D mata en las nativas con HR similar al global, el '
      'hallazgo central es propio de la dinamica de la conversacion y no un '
      'artefacto de calendario de resultados; (2) la p de la interaccion dice '
      'si la mecanica de muerte difiere entre poblaciones; (3) el dummy '
      'earnings resume si los eventos institucionales mueren mas rapido '
      'controlando todo lo demas (la mediana 7 vs 11 ya lo sugiere).')

etiqueta censal: {'sin_earnings': 1818, 'sin_rdq': 572, 'earnings': 401}
eventos en el contraste (con mercado completo): earnings 384 | nativos 1,724 | excluidos sin etiqueta: 528
KM earnings: mediana 6 dias | sobrevive 30d: 6.5% (n=384)
KM nativos: mediana 11 dias | sobrevive 30d: 11.9% (n=1,724)
log-rank earnings vs nativos: p = 2.04e-23

=== P1 - Cox integrado I2, poblacion NATIVA (sin earnings) ===
                      coef  exp(coef)  exp(coef) lower 95%  exp(coef) upper 95%       p
covariate                                                                              
b_duro_lag         -0.1336     0.8749               0.7891               0.9701  0.0112
d_duro_lag          0.4701     1.6001               1.1525               2.2218  0.0050
sin_direccion_lag   0.7826     2.1872               1.7702               2.7023  0.0000
log1p_n_lag         0.4786     1.6138               1.5318               1.7001  0.0000
log_z_inicio        0.3063     1.3583               1.2361        

In [1]:
exec(open('celda_S8.py').read())


etiqueta v2 censal: {'nativo': 1517, 'sin_rdq': 572, 'earnings': 401, 'institucional_no_earnings': 202, 'sin_cobertura_prensa': 99}
eventos en el contraste (con mercado completo): {'nativo': 1438, 'institucional_no_earnings': 191, 'earnings': 384} | excluidos (sin_rdq/sin_cobertura): 623
KM nativo: mediana 12 dias | sobrevive 30d: 11.8% (n=1,438)
KM institucional_no_earnings: mediana 9 dias | sobrevive 30d: 7.3% (n=191)
KM earnings: mediana 6 dias | sobrevive 30d: 6.5% (n=384)
log-rank global (3 poblaciones): p = 3.14e-25
log-rank nativo vs institucional_no_earnings: p = 6.01e-07
log-rank institucional_no_earnings vs earnings: p = 1.18e-02

=== Q1 - Cox I2, poblacion NATIVA v2 (sin earnings ni ola de prensa) ===
                      coef  exp(coef)  exp(coef) lower 95%  exp(coef) upper 95%       p
covariate                                                                              
b_duro_lag         -0.1595     0.8526               0.7648               0.9505  0.0040
d_duro_lag    